In [1]:
#!pip install pandas pyarrow
# import sys
# !{sys.executable} -m pip install pyarrow

In [2]:
## ENSG as key column
## then one column for each filter subgroup 
## will keep UniProt column for now

# quartile data
## NEFF - by uniprot
## proportion disorder - by uniprot
## protein length - by uniprot

# non-quartile
## constraint (10-90 and 20-80) - ENSG
## percent repeat (3 classes) - by uniprot
## CATH class and architecture - by uniprot (can i get this by ENSG on the redo?) ***** skipping for now
## dominant/recessive - by gene name and uniprot (use uniprot)
## kinases - by gene name and uniprot (use uniprot)
## GCPRs - by gene name and uniprot (use uniprot)
## surface - ENSG
## brain expression - ENSG
## enzymes - by uniprot

# during this process
## create a table of counts of genes included - are we losing data unexpectedly?
## (beyond uniprot IDs that do not map to ENSG or that did not exist in a given version)

In [3]:
import pandas as pd
import numpy as np

In [4]:
def filter_files_quartiles(df, ps_df, ps_col, feature, id_col, df_c, missing):
    # id_col must be 'uniprot_id' or 'ensg'
    # subset list
    subset_list = [f"{feature}_q1_low", f"{feature}_q2", f"{feature}_q3", f"{feature}_q4_high"]
    # quartiles
    ps_df[f"{ps_col}_quartile"] = pd.qcut(
        ps_df[ps_col],
        q=4,
        labels=subset_list
    )
    all_ids = ps_df[id_col].unique()
    # how many ids shared between tables
    id_c = len(df[df[id_col].isin(all_ids)])
    df_c.loc[len(df_c)] = {'subset': feature, 'ID column': id_col, 'initial count': len(all_ids), 'final count': id_c}
    # add to missing list
    missing.update(set(all_ids) - set(df[df[id_col].isin(all_ids)][id_col]))
    # get list of proteins (by ENSG) in each class
    # add that as column to main dataframe
    for d in subset_list:
        #print(d)
        sub_ids = ps_df[ps_df[f"{ps_col}_quartile"]==d][id_col].unique()
        #  need genes not in ps_df to be NA
        df[d] = pd.NA # initialize with NA
        df.loc[df[id_col].isin(sub_ids), d] = True # True where ensg is in ensg_ids
        #df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), d] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [5]:
# base table that we are building everything off of:
df = pd.read_parquet('gs://genetics-gym/linkers/linker_ensg_uniprot_dedup.parquet')
# columns: ensg & uniprot_id (not isoform)

# table to track counts:
df_counts = pd.DataFrame(columns=['subset', 'ID column', 'initial count', 'final count'])

# missing uniprot
miss_u = set()

# missing ensg
miss_e = set()

In [6]:
# Quartile data
# Proportion disordered
feature = 'proportion_disordered'
ps_path = '../disorder_proportions.tsv' # protein subset data path; should be a tsv file
ps_col = 'proportion_disordered' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t')
filter_files_quartiles(df, ps_df, ps_col, feature, 'uniprot_id', df_counts, miss_u)
# protein length
feature = 'protein_length'
ps_path = '../../reference_data/uniprot_af2db_isoform_guide/uniprot_summary_11_13_2025.csv' # protein subset data path; should be a tsv file
ps_col = 'sequence_length' # column of relevant data
ps_df = pd.read_csv(ps_path)
filter_files_quartiles(df, ps_df, ps_col, feature, 'uniprot_id', df_counts, miss_u)
# NEFF
feature = 'neff'
ps_col = 'neff'
ps_path = '../gnomad/gnomad_by_protein/gnomad_neff/neff_result.csv'
ps_df = pd.read_csv(ps_path)
filter_files_quartiles(df, ps_df, ps_col, feature, 'uniprot_id', df_counts, miss_u)

In [7]:
# constraint
feature = 'constraint'
ps_path = 'all_genes_scores_LEOUF.tsv'
ps_col = 'LOEUF' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t')
id_col = 'ensg'
# subset list
subset_list = [f"{feature}_top_10", f"{feature}_top_20", f"{feature}_bottom_90", f"{feature}_bottom_80"]
# determine subsets
top_10_cutoff = ps_df[ps_col].quantile(0.9)
top_20_cutoff = ps_df[ps_col].quantile(0.8)
ps_df['top_10'] = ps_df[ps_col]>=top_10_cutoff
ps_df['top_20'] = ps_df[ps_col]>=top_10_cutoff

all_ids = ps_df[id_col].unique()
# how many ids shared between tables
id_c = len(df[df[id_col].isin(all_ids)]['ensg'].unique())
df_counts.loc[len(df_counts)] = {'subset': feature, 'ID column': id_col, 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_e.update(set(all_ids) - set(df[df[id_col].isin(all_ids)][id_col]))

# get list of proteins (by ENSG) in each class
# add that as column to main dataframe

sub_ids = ps_df[ps_df.top_10].ensg.unique()
df[f"{feature}_top_10"] = pd.NA # initialize with NA
df.loc[df[id_col].isin(sub_ids), f"{feature}_top_10"] = True # True where ensg is in ensg_ids
#df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), f"{feature}_top_10"] = False # False where ensg NOT in ensg_ids but IS in all_ensg

sub_ids = ps_df[~ps_df.top_10].ensg.unique()
df[f"{feature}_bottom_90"] = pd.NA # initialize with NA
df.loc[df[id_col].isin(sub_ids), f"{feature}_bottom_90"] = True # True where ensg is in ensg_ids
#df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), f"{feature}_bottom_90"] = False # False where ensg NOT in ensg_ids but IS in all_ensg

sub_ids = ps_df[ps_df.top_20].ensg.unique()
df[f"{feature}_top_20"] = pd.NA # initialize with NA
df.loc[df[id_col].isin(sub_ids), f"{feature}_top_20"] = True # True where ensg is in ensg_ids
#df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), f"{feature}_top_20"] = False # False where ensg NOT in ensg_ids but IS in all_ensg

sub_ids = ps_df[~ps_df.top_20].ensg.unique()
df[f"{feature}_bottom_80"] = pd.NA # initialize with NA
df.loc[df[id_col].isin(sub_ids), f"{feature}_bottom_80"] = True # True where ensg is in ensg_ids
#df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), f"{feature}_bottom_80"] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [8]:
# percent repeat
feature = 'repeats' # protein subset - specific
ps_path = '../gnomad/gnomad_by_protein/gnomad_repeats/uniprot_repeats.tsv' # protein subset data path; should be a tsv file
ps_col = 'percent_in_repeat' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t')
id_col = 'uniprot_id'
## groups: 0, 0-10, >=10
bins = [-np.inf, 0, 10, np.inf]
subset_list = ['0_percent_repeat', '0_10_percent_repeat', 'gt_10_percent_repeat']
ps_df['repeat_percent_group'] = pd.cut(
    ps_df[ps_col],
    bins=bins,
    labels=subset_list,
    right=True,          
    include_lowest=True
)

all_ids = ps_df[id_col].unique()
# how many ids shared between tables
id_c = len(df[df[id_col].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': feature, 'ID column': id_col, 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df[id_col].isin(all_ids)][id_col]))

# get list of proteins in each class
# add that as column to main dataframe

## save relevant filter files
for d in subset_list:
    sub_ids = ps_df[ps_df['repeat_percent_group']==d][id_col].unique()
    #  need genes not in ps_df to be NA
    df[d] = pd.NA # initialize with NA
    df.loc[df[id_col].isin(sub_ids), d] = True # True where ensg is in ensg_ids
    #df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), d] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [9]:
# dominant/recessive
dom = pd.read_csv('../macarthur_gene_lists_all_ad.tsv', sep='\t', names=['gene'])
rec = pd.read_csv('../macarthur_gene_lists_all_ar.tsv', sep='\t', names=['gene'])
# add in uniprot id - using scan test protein guide
#uni_df = pd.read_csv('../../scan_test_project/gene_uniprot_guide/gene_to_uniprot_id.tsv', sep='\t')
uni_df = pd.read_csv('../../scan_test_project/all_missense_variants_2_2026/protein_sequence_guide.tsv', sep='\t')
uni_df = uni_df.rename(columns={'gene_name': 'gene_symbol'})
dom = dom.merge(uni_df[['uniprot_id', 'gene_symbol']], left_on='gene', right_on='gene_symbol', how='left')
dom = dom[['gene', 'uniprot_id']]
rec = rec.merge(uni_df[['uniprot_id', 'gene_symbol']], left_on='gene', right_on='gene_symbol', how='left')
rec = rec[['gene', 'uniprot_id']]
# filling in missing genes
genes_more = {'AARS': 'P49588', 'DFNA5': 'O60443', 'FGFR1OP': 'O95684', 'GARS': 'P41250', 
              'KIAA0196': 'Q12768', 'LOR': 'P23490', 'SHFM1': 'P60896', 'SLC9A3R1': 'O14745',
              'TDGF1': 'P13385', 'YARS': 'P54577', 'ADCK3': 'Q8NI60', 'B3GALTL': 'Q6Y288', 
              'C12orf65': 'Q9H3J6', 'C14orf166': 'Q9Y224', 'C2orf71': 'A6NGG8', 'CIRH1A': 'Q969X6', 
              'DFNB31': 'Q9P202', 'FAM126A': 'Q9BYI3', 'FAM134B': 'Q9H6L5', 'FAM188A': 'Q9H8M7', 
              'G6PC': 'P35575', 'GBA': 'P04062', 'GIF': 'P27352', 'GPR56': 'Q9Y653', 'HFE2': 'Q6ZVN8', 
              'ICK': 'Q9UPZ9', 'IKBKAP': 'O95163', 'KAAG1': 'Q9UBP8', 'KARS': 'Q15046', 
              'KIAA1279': 'Q96EK5', 'LARGE': 'O95461', 'LRTOMT': 'Q8WZ04', 'MRE11A': 'P49959', 
              'MUT': 'P22033', 'PARK2': 'O60260', 'PTRF': 'Q6NZI2', 'PVRL1': 'Q15223', 
              'PVRL4': 'Q96NY8', 'SEPN1': 'Q9NZV5', 'SPG20': 'Q8N0X7', 'WISP3': 'O95389', 
              'SIK1B': 'P57059', 'ADGRF2': 'Q8IZF7', 'GPR1': 'P46091', 'OR10AC1': 'Q8NH08', 
              'OR10J4': 'P0C629', 'OR11H7': 'Q8NGC8', 'OR12D1': 'P0DN82', 'OR1D4': 'P47884', 
              'OR1E3': 'Q8WZA6', 'OR1P1': 'Q8NH06', 'OR2T7': 'P0C7T2', 'OR2W5': 'A6NFC9', 
              'OR4A8': 'P0C604', 'OR4C45': 'A6NMZ5', 'OR4K3': 'Q96R72', 'OR4Q2': 'P0C623', 
              'OR51J1': 'Q9H342', 'OR52E1': 'Q8NGJ3', 'OR5AC1': 'P0C628', 'OR5G3': 'P0C626', 
              'OR5R1': 'Q8NH85', 'OR8J2': 'Q8NGG1', 'VN1R3': 'Q9BXE9', 'VN1R5': 'Q7Z5H4'}
dom['uniprot_id'] = dom['uniprot_id'].fillna(dom['gene'].map(genes_more))
rec['uniprot_id'] = rec['uniprot_id'].fillna(rec['gene'].map(genes_more))

all_ids = pd.concat([dom, rec]).uniprot_id.unique()
# how many ids shared between tables
id_c = len(df[df['uniprot_id'].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': 'dominant/recessive', 'ID column': 'uniprot_id', 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df['uniprot_id'].isin(all_ids)]['uniprot_id']))

dom_ids = dom.uniprot_id.unique()
rec_ids = rec.uniprot_id.unique()

df['dominant'] = pd.NA # initialize with NA
df.loc[df['uniprot_id'].isin(dom_ids), 'dominant'] = True # True where ensg is in ensg_ids
#df.loc[~df['uniprot_id'].isin(dom_ids) & df['uniprot_id'].isin(rec_ids), 'dominant'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

df['recessive'] = pd.NA # initialize with NA
df.loc[df['uniprot_id'].isin(rec_ids), 'recessive'] = True # True where ensg is in ensg_ids
#df.loc[~df['uniprot_id'].isin(rec_ids) & df['uniprot_id'].isin(dom_ids), 'recessive'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [10]:
# kinases
kin = pd.read_csv('kinases.tsv', sep='\t', names=['gene'])
kin = kin.merge(uni_df[['uniprot_id', 'gene_symbol']], left_on='gene', right_on='gene_symbol', how='left')
kin = kin[['gene', 'uniprot_id']]
kin['uniprot_id'] = kin['uniprot_id'].fillna(kin['gene'].map(genes_more))

all_ids = kin.uniprot_id.unique()
# how many ids shared between tables
id_c = len(df[df['uniprot_id'].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': 'kinases', 'ID column': 'uniprot_id', 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df['uniprot_id'].isin(all_ids)]['uniprot_id']))

df['kinase'] = pd.NA # initialize with NA
df.loc[df['uniprot_id'].isin(all_ids), 'kinase'] = True # True where ensg is in ensg_ids
#df.loc[~df['uniprot_id'].isin(all_ids) & df['uniprot_id'].isin(all_ids), 'kinase'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [11]:
# GCPRs
gcprs = pd.read_csv('gpcr_union.tsv', sep='\t', names=['gene'])
gcprs = gcprs.merge(uni_df[['uniprot_id', 'gene_symbol']], left_on='gene', right_on='gene_symbol', how='left')
gcprs = gcprs[['gene', 'uniprot_id']]
gcprs['uniprot_id'] = gcprs['uniprot_id'].fillna(gcprs['gene'].map(genes_more))

all_ids = gcprs.uniprot_id.unique()
# how many ids shared between tables
id_c = len(df[df['uniprot_id'].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': 'gcprs', 'ID column': 'uniprot_id', 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df['uniprot_id'].isin(all_ids)]['uniprot_id']))

df['gcpr'] = pd.NA # initialize with NA
df.loc[df['uniprot_id'].isin(all_ids), 'gcpr'] = True # True where ensg is in ensg_ids
#df.loc[~df['uniprot_id'].isin(all_ids) & df['uniprot_id'].isin(all_ids), 'gcpr'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [12]:
# surface proteins
# download source: https://wlab.ethz.ch/surfaceome/
surface = pd.read_excel('table_S3_surfaceome.xlsx', skiprows=1)
surface_ids = surface[surface['Surfaceome Label']=='surface']['Ensembl gene'].unique()

# how many ids shared between tables
id_c = len(df[df['ensg'].isin(surface_ids)]['ensg'].unique())
df_counts.loc[len(df_counts)] = {'subset': 'surface proteins', 'ID column': 'ensg', 'initial count': len(surface_ids), 'final count': id_c}
# add to missing list
miss_e.update(set(surface_ids) - set(df[df['ensg'].isin(surface_ids)]['ensg']))

df['surface_protein'] = pd.NA # initialize with NA
df.loc[df['ensg'].isin(surface_ids), 'surface_protein'] = True # True where ensg is in ensg_ids
#df.loc[~df['ensg'].isin(surface_ids) & df['ensg'].isin(surface_ids), 'surface_protein'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [13]:
# high brain expression
# download source: https://gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
be = pd.read_csv("GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz", sep='\t', skiprows=2, compression='gzip')
be['ensg'] = [x.split('.')[0] for x in be.Name]
be['brain_expression'] = be['Brain_Cortex']
be = be[['ensg', 'brain_expression']]
cutoff = np.quantile(be.brain_expression.dropna(), 0.9)
be_ids = be[be.brain_expression>cutoff].ensg.unique()

# how many ids shared between tables
id_c = len(df[df['ensg'].isin(be_ids)]['ensg'].unique())
df_counts.loc[len(df_counts)] = {'subset': 'high brain expression proteins', 'ID column': 'ensg', 
                                 'initial count': len(be_ids), 'final count': id_c}
# add to missing list
miss_e.update(set(be_ids) - set(df[df['ensg'].isin(be_ids)]['ensg']))

df['high_brain_expression_protein'] = pd.NA # initialize with NA
df.loc[df['ensg'].isin(be_ids), 'high_brain_expression_protein'] = True # True where ensg is in ensg_ids
#df.loc[~df['ensg'].isin(be_ids) & df['ensg'].isin(be_ids), 'high_brain_expression_protein'] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [14]:
# enzymes
# search "organism_id:9606 AND ec:*" on UniProt and download results
e = pd.read_csv('uniprotkb_organism_id_9606_AND_ec_2026_04_09.tsv.gz', sep='\t', compression='gzip')
e = e.rename(columns={'Entry': 'uniprot_id'})
e_ids = e.uniprot_id.unique()
# how many ids shared between tables
id_c = len(df[df['uniprot_id'].isin(e_ids)])
df_counts.loc[len(df_counts)] = {'subset': 'enzymes', 'ID column': 'uniprot_id',
                                 'initial count': len(e_ids), 'final count': id_c}
# add to missing list
#miss_u.update(set(e_ids) - set(df[df['uniprot_id'].isin(e_ids)]['uniprot_id']))
# majority of uniprot IDs unreviewed (won't be in linker tables)
## 21670 out of 26274

df['enzyme'] = pd.NA # initialize with NA
df.loc[df['uniprot_id'].isin(e_ids), 'enzyme'] = True # True where ensg is in ensg_ids
#df.loc[~df['uniprot_id'].isin(e_ids) & df['uniprot_id'].isin(e_ids), 'high_brain_expression_protein'] = False # False where ensg NOT in ensg_ids but IS in all_ensg
# this is pulled from uniprot so technically all uniprot ids are valid for this (no NAs?)

In [15]:
# CATH Class
feature = 'cath_class' # protein subset - general
ps_path = '../domains/uniprot_classes_clean.tsv' # protein subset data path; should be a tsv file
ps_col = 'CATH_class' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t')
id_col = 'uniprot_id'
## EXTRA STEP drop anything with multiple classes
all_cath_ids = set(ps_df.uniprot_id)
ps_df = ps_df[ps_df.groupby("uniprot_id")["CATH_class"].transform("nunique") == 1]
ps_df = ps_df.drop_duplicates(subset='uniprot_id')
## get list of no-cath proteins (everything else in uniprot
uniprot_df = pd.read_csv('../../reference_data/uniprot_af2db_isoform_guide/uniprot_summary_11_13_2025.csv')
uni_ids = set(uniprot_df.uniprot_id.unique())
nocath = list(uni_ids - all_cath_ids)

all_ids = ps_df[id_col].unique()
# how many ids shared between tables
id_c = len(df[df[id_col].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': feature, 'ID column': id_col,
                                 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df[id_col].isin(all_ids)][id_col]))

## subsets
subset_list = list(ps_df.CATH_class.unique().astype(str))
subset_list.append('None')

for d in subset_list:
    d_name = f"CATH_class_{str(d)}"
    if d == 'None':
        #  need genes not in ps_df to be NA
        df[d_name] = pd.NA # initialize with NA
        df.loc[df[id_col].isin(nocath), d_name] = True # True where ensg is in ensg_ids
    else: 
        sub_ids = ps_df[ps_df[ps_col]==int(d)][id_col].unique()
        #  need genes not in ps_df to be NA
        df[d_name] = pd.NA # initialize with NA
        df.loc[df[id_col].isin(sub_ids), d_name] = True # True where ensg is in ensg_ids
        #df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), d] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [16]:
# CATH Architecture
feature = 'cath_arch' # protein subset - general
ps_path = '../domains/uniprot_architecture_clean.tsv' # protein subset data path; should be a tsv file
ps_col = 'CATH_architecture' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t', dtype={ps_col: str})
id_col = 'uniprot_id'
## EXTRA STEP drop anything with multiple architectures
all_cath_ids = set(ps_df.uniprot_id)
ps_df = ps_df[ps_df.groupby("uniprot_id")["CATH_architecture"].transform("nunique") == 1]
ps_df = ps_df.drop_duplicates(subset='uniprot_id')
## get list of no-cath proteins (everything else in uniprot
uniprot_df = pd.read_csv('../../reference_data/uniprot_af2db_isoform_guide/uniprot_summary_11_13_2025.csv')
uni_ids = set(uniprot_df.uniprot_id.unique())
nocath = list(uni_ids - all_cath_ids)

all_ids = ps_df[id_col].unique()
# how many ids shared between tables
id_c = len(df[df[id_col].isin(all_ids)])
df_counts.loc[len(df_counts)] = {'subset': feature, 'ID column': id_col,
                                 'initial count': len(all_ids), 'final count': id_c}
# add to missing list
miss_u.update(set(all_ids) - set(df[df[id_col].isin(all_ids)][id_col]))

## subsets
subset_list = list(ps_df.CATH_architecture.unique().astype(str))
subset_list.append('None') 

for d in subset_list:
    d = str(d)
    d = d.replace('/', '-')
    d = d.replace(':', '_')
    d = d.replace('.', '_')
    d_name = f"CATH_architecture_{str(d)}"
    if d == 'None':
        #  need genes not in ps_df to be NA
        df[d_name] = pd.NA # initialize with NA
        df.loc[df[id_col].isin(nocath), d_name] = True # True where ensg is in ensg_ids
    else: 
        sub_ids = ps_df[ps_df[ps_col]==d][id_col].unique()
        #  need genes not in ps_df to be NA
        df[d_name] = pd.NA # initialize with NA
        df.loc[df[id_col].isin(sub_ids), d_name] = True # True where ensg is in ensg_ids
        #df.loc[~df[id_col].isin(sub_ids) & df[id_col].isin(all_ids), d] = False # False where ensg NOT in ensg_ids but IS in all_ensg

In [17]:
df

,ensg,uniprot_id,proportion_disordered_q1_low,proportion_disordered_q2,proportion_disordered_q3,proportion_disordered_q4_high,protein_length_q1_low,protein_length_q2,protein_length_q3,protein_length_q4_high,...,CATH_architecture_1_50,CATH_architecture_3_70,CATH_architecture_2_100,CATH_architecture_3_75,CATH_architecture_2_110,CATH_architecture_3_15,CATH_architecture_3_100,CATH_architecture_3_55,CATH_architecture_2_102,CATH_architecture_None
0,ENSG00000127054,E9PIG1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,ENSG00000008128,J3KS35,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,ENSG00000008130,A0A0A0MR98,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,ENSG00000142609,Q9C0B2,<NA>,<NA>,True,<NA>,True,<NA>,<NA>,True,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,ENSG00000130762,Q5VV41,<NA>,<NA>,True,<NA>,<NA>,True,<NA>,True,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65067,ENSG00000125733,W4VSQ9,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
65068,ENSG00000142459,M0R1Y1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
65069,ENSG00000183248,A0A096LNU1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
65070,ENSG00000171017,M0QZ48,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [18]:
df_counts

,subset,ID column,initial count,final count
0,proportion_disordered,uniprot_id,19573,16589
1,protein_length,uniprot_id,20420,16434
2,neff,uniprot_id,20168,16220
3,constraint,ensg,21849,16290
4,repeats,uniprot_id,20420,16434
5,dominant/recessive,uniprot_id,1854,1653
6,kinases,uniprot_id,346,294
7,gcprs,uniprot_id,759,650
8,surface proteins,ensg,2721,2185
9,high brain expression proteins,ensg,7463,6057


In [19]:
len(miss_u)

4453

In [ ]:
# does not map to ensemble (31):
## P01893, Q5T3Y7, P60509, P0CG43, Q99440, P0DMU2, O15255, Q9BWJ2, Q6NVV9, Q6ZN68, P43359, Q8NGV5, P0CF51,
## Q6ZSR9, Q63HN1, Q6ZRV3, Q96P88, P0C879, P01880, A4QPB2, Q8N1I8, Q8N7Q2, Q96PT3, Q32M92, Q495Z4, Q8IVE0,
## A6NM43, P63127, Q5T035, P61575, Q8NBR9

# corresponding ENSG did not exist in ensembl v105 (2):
## Q9H227 (ENSG00000291927); P61550 (ENSG00000293570)

# unreviewed UniProt ID (5):
## A0A075B6W8, F6TER3, A0A8I5KXN5, A0A8V8TLY5, A0A9L9PY38

# uniprot ID did not exist (or was unreviewed) before December 2021 (ensembl v105 released then) (2):
## P0DQW1, A0A411D538

# uniprot says not on autosomes + X (12):
## P24071, P00403, Q9P244, Q96HB5, Q9H313, Q9P1W9, Q6L8H1, P43365, O60831, P03886, Q5T3I0, Q03923

# no variants in uniprot (1):
## P0C629
### not sure if this is a real reason but seems suspicious

## 52 explained
## 97 unexplained but looked at
## out of 4453

# Q3LI61 - ENSG00000184032 also missing - ****
# Q9HA65 - ENSG00000104946 also missing - ****
# Q96D98 - ENSG00000176401 also missing - ****
# P11413 - ENSG00000160211 also missing - has a lot of computationally mapped isoforms? - ****
# Q16617 - ENSG00000105374 also missing - ****
# A0A2R8Y7Y5 - ENSG00000284797 also missing - ****
# Q569K6 - ENSG00000187860 also missing - ****
# Q14498 - ENSG00000131051 also missing - ****
# Q8N2S1 - ENSG00000090006 also missing - ****
# Q9NPB8 - ENSG00000125772 - also missing - ****
# Q9UPP1 - ENSG00000172943 also missing - ****
# Q9HCS2 - ENSG00000186204 also missing - ****
# Q6ZPD8 - ENSG00000184210 also missing - ****
# P51808, Q6ZRI8, Q8N6L0, Q7Z406, Q8N3F8, Q9Y2G7, Q494X3, P31997, Q99426, P28325, A1L0T0, Q8NFJ6, Q06187, Q96MZ0
# Q86X51, Q9BSG5, P0C0E4, P05362, Q70EK9, P0CG32, Q99687, Q96GP6, P0DTW1, P15814, P29965, Q2M3W8, Q96GC6, Q14656, 
# P20749, P35240, Q9BY66, Q96GY3, Q8N319, Q9UJM8, Q9BXH1, Q9NRZ7, Q9H321, Q13356, Q5JUK9, P62318, Q96LI9, Q16720, 
# P09601, P53814, Q5JWF2, Q92841, O15550, P0DN86, Q9ULM2, Q96IZ5, P09086, Q9UKY0, Q9Y3M9, P60413, P58511, H3BPF8, 
# P51654, Q9HDC9, Q9UP65, O43555, Q3SXP7, Q8TBC3, P49223, P0DMW4, Q8TB03, Q9HAP6, Q96RL6, O76054, P59826, Q96S44, 
# Q96Q04, P19971, Q9UN88, Q6UXV1, P56537, Q9NSD4, Q6ICG8, P25685, Q9UN30, Q9NPA3, Q9ULX6, Q96RP8, Q96DF8, Q6ICB4
## and many more

In [40]:
df[df.ensg=='ENSG00000184210']

,ensg,uniprot_id,proportion_disordered_q1_low,proportion_disordered_q2,proportion_disordered_q3,proportion_disordered_q4_high,protein_length_q1_low,protein_length_q2,protein_length_q3,protein_length_q4_high,...,CATH_architecture_1_50,CATH_architecture_3_70,CATH_architecture_2_100,CATH_architecture_3_75,CATH_architecture_2_110,CATH_architecture_3_15,CATH_architecture_3_100,CATH_architecture_3_55,CATH_architecture_2_102,CATH_architecture_None


In [20]:
# enzymes pushes that number up to 20,000+ - need to look into that one specifically
# but start with this list for now

In [21]:
len(miss_e)

5958

In [27]:
# save to tsv file
df.to_csv('ensg_filters.tsv', sep='\t', index=False)